In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if any(importlib.util.find_spec(name) is None
       for name in ("adh_ai_labs", "transformers", "sentence_transformers")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "adh-ai-labs[text] @ git+https://github.com/BradSegal/OxfordADH2026AILabs.git"], check=True)
    importlib.invalidate_caches()
print("Course helpers ready.")

# How a language model reads and writes

**MSc in Applied Digital Health.** About 45 mins.

You have used a large language model (LLM) via a chat website: you type and a response comes back. In this lab you will open one up and manually run each step on the CPU of this Colab machine. The same ECG machine reports will be used as in notebook 01.

| Part | Time | What you do |
|---|---|---|
| Set-up | 6 min | Load three small models and reports |
| 1 | 3 min | Watch how text is tokenised into the pieces the model reads |
| 2 | 10 min | Learn what a vector is, convert text to vectors, and understand the similarity between meanings (TODO 1) |
| 3 | 13 min | Search within reports by meaning and identify where it fails (TODOs 2 and 3) |
| 4 | 12 min | Convert the model’s scores to probabilities and write the generation loop (TODOs 4 and 5) |
| 5 | 5 min | See that a chat is one long text the model continues |

### How this notebook works

- Each **TODO** cell needs 1-3 lines of code, with some code partially written for you. You just need to replace each `...`. The check below will print ✅ if your code is correct, ❌ with a hint if it's not, or ⏳ if you have left a `...` in your code.
- Each **Try it** cell has a text box or slider on the right-hand side. Change this, then run the cell. There is no right answer and no check.
- **Check your understanding** by writing your answer between the quotes, e.g. `check("3a", "c")`, and running the cell.

### Before you start

1. **Data:** the same zip as notebook 01, `adh_multimodal_lab_data.zip`, located in `MyDrive/adh_multimodal_lab/` on Google Drive. Please link your Drive when requested.
2. No GPU required: all models in this lab are small enough to be run on the CPU.
3. Make a copy of this notebook to edit: *File > Save a copy in Drive*.

The machine reports are real and sourced from the open MIMIC-IV-ECG database (ODbL licence). Please keep these reports confidential to the course and do not paste these into an AI chatbot. This is also the reason why every model in this lab runs on this machine: no report is sent anywhere.

## Set-up

The next cell loads the libraries and the three models. The first time you run it, it will download about 1.5 GB, which may take a minute or two.

In [ ]:
# @title Load the libraries and the three models { display-mode: "form" }
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from adh_ai_labs import data, language, language_figures
from adh_ai_labs.checks import llm_applications as checks
from adh_ai_labs.checks.llm_applications import check

lm = language.load()

- The language model we’ll use for writing text is **Qwen2.5 0.5B Instruct** (Alibaba). It has about 500 million **parameters** (think of these as the numbers the model has ‘learnt’ through training, like the weights from notebook 01).
- **all-MiniLM-L6-v2**: a general-purpose **embedding model** for transforming text into a numerical representation of the meaning of text.
- **PubMedBERT embeddings**: a similar embedding model trained on biomedical text.

### Where the lab is going

In [ ]:
language_figures.roadmap()

### The reports

In [ ]:
meta = data.load_metadata()
reports = language.unique_reports(meta)
print(f"{len(meta):,} ECGs, {len(reports)} different reports. Two of them:")
for report in reports[4:6]:
    print("  ", report)

## Part 1: text becomes tokens (3 min)

Text must first be converted to numbers for a model to work with. This is done by a **tokeniser** which splits text into **tokens** from a fixed list of elements called a **vocabulary**. Each token has an associated identifier (id) that indicates the position of it within the vocabulary list. Qwen in this case has a vocabulary of 151,665 tokens which were selected to be as common as possible in the text from the model’s training. A common word will typically be a single token; however, if the word is very rare it will be split into multiple tokens.

In [ ]:
report = "Sinus bradycardia with 1st degree A-V block. Otherwise normal ECG"
language_figures.token_boxes(lm.pieces(report), lm.tokens(report))

‘with’, ‘degree’, ‘block’, and ‘normal’ are all single tokens (with the space included). ‘bradycardia’ is 5 tokens, and ‘ECG’ is 2. The model never sees the word ‘bradycardia’; it only sees the ids of the split word pieces.

### Try it: the tokeniser

Type anything you want in the box below and run the cell. Try a drug name, a misspelling, a number like 145, and even a sentence in a different language. See where the cuts are made. Note that some tokens include parts of characters (this is common for some languages and emojis) and so may appear as �. If you decode the tokens together you’ll get back the original text.

In [ ]:
MY_TEXT = "Sinus bradycardia with 1st degree A-V block"  # @param {type:"string"}
language_figures.token_boxes(lm.pieces(MY_TEXT), lm.tokens(MY_TEXT))

Tokens are important for two main reasons:

- The model can only read a set number of tokens at a time: this is called the **context window**.
- Models that are hosted charge for usage by the token.

Standard English is roughly one token per word; however, clinical words tend to be split into multiple tokens. This means that clinical text is more expensive to process. These tokenisers are model specific.

## Part 2: meaning becomes a vector (10 min)

### What a vector is

In notebook 01 we reduced each ECG to two numbers, the heart rate and RR irregularity, and then plotted them on a graph. A **vector** is simply a list of numbers in a specific order. We can visualise this as an arrow from (0, 0) to wherever the point is. This arrow has a **length** (how long it is) and a **direction** (where it points towards).

In [ ]:
language_figures.ecg_vectors()

After scaling, the (0, 0) point is the average ECG. The arrows now point to how the specific ECGs differ from the average. Both regular rhythms point down due to the fact that their RR irregularity is less than average, bradycardia left and tachycardia right due to their heart rate being less or greater than average. The AF vector points to the top indicating that it is irregular. The direction is therefore the type of ECG and the length is how different it is from the average.

### Text as a vector

An embedding model achieves a similar goal for text, except leveraging far more numbers. It transforms an entire text piece into a single vector. For MiniLM this represents 384 numbers, for PubMedBERT, 768. The actual meaning of each of these numbers is not explicitly defined but is learnt as a property of training the model such that text with similar meanings has vectors that point in a similar direction.

`language.embed` takes a list of texts and returns a single row of numbers for each; `[0]` retrieves the first row.

In [ ]:
vector = language.embed(["atrial fibrillation"], "biomedical")[0]
print(f"{len(vector)} numbers. The first six: {vector[:6].round(3)}")

A single number is fairly meaningless: it is the direction of the overall arrow that provides meaning.

### Where clinical terms land

The next cell demonstrates the embedding of 33 different clinical terms. These range from clinical to lay terms, abbreviations, and both brand and generic drug names. For visualisation of the 384 or 768 numbers, we apply a dimensionality reduction technique known as **PCA** (see the end of notebook 01) to find the two axes of greatest variation in the data and project the points onto these axes. A dotted line is made for points that are synonyms. The shorter the line, the more similar the model thinks the terms are.

In [ ]:
terms, groups = list(language.CLINICAL_TERMS), list(language.CLINICAL_TERMS.values())
maps = {}
for model in ["general", "biomedical"]:
    vectors = language.embed(terms, model)
    maps[f"{model} model ({vectors.shape[1]} numbers per term)"] = PCA(n_components=2).fit_transform(vectors)
language_figures.term_map(maps, terms, groups, language.SAME_MEANING)

What is the spatial distribution of terms within each model? How are abbreviations or drug names represented? While there is clear clustering of similar terms, there are clearly deficiencies in each model.

### Comparing directions: cosine similarity

The method of comparing meaning is by comparing the direction of the vector, i.e., the angle between them. This is achieved using **cosine similarity**, which calculates the cosine of the angle between the vectors. This provides a value of 1 if the two vectors are in the same direction, 0 if they are orthogonal (at right angles), and -1 if they are opposite.

In [ ]:
language_figures.cosine_arrows()

Given two vectors $a$ and $b$ and the angle $\theta$ between them:

$$\cos\theta = \frac{a \cdot b}{\|a\|\,\|b\|}$$

- $a \cdot b = a_1 b_1 + a_2 b_2 + \dots$ is the **dot product**, where we multiply matching numbers, then add them up.
- $\|a\| = \sqrt{a \cdot a}$ is the **length** of $a$.
- The division by both lengths ensures only the direction is being measured.

Manual calculation for $a = [1, 2, 2]$ and $b = [2, 1, 2]$:

- dot product: $1 \times 2 + 2 \times 1 + 2 \times 2 = 8$
- lengths: $\|a\| = \sqrt{1 + 4 + 4} = 3$ and $\|b\| = 3$
- cosine similarity: $8 / (3 \times 3) = 0.89$

### TODO 1: cosine similarity

Complete the two lines of `cosine(a, b)`.

- `dot`: the dot product. In NumPy, `a @ b` is the dot product of `a` and `b`.
- `lengths`: the two lengths multiplied. `np.linalg.norm(a)` is the length of `a`.

When the check mark appears (✅), your function will be compared to `cosine_similarity` from scikit-learn, the library one-liner.

In [ ]:
def cosine(a, b):
    """The cosine similarity of two vectors."""
    dot = ...  # the dot product of a and b
    lengths = ...  # the length of a times the length of b
    return dot / lengths


if checks.cosine(cosine):
    a, b = np.array([1, 2, 2]), np.array([2, 1, 2])
    print(f"Yours: {cosine(a, b):.2f}. scikit-learn: {cosine_similarity([a], [b])[0, 0]:.2f}")

### Try it: compare two phrases

Insert any two phrases and run the cell to compare the cosine similarity output from each model. Try a brand vs generic name, a lay vs clinical term, a misspelling, or two languages.

In [ ]:
PHRASE_1 = "shortness of breath"  # @param {type:"string"}
PHRASE_2 = "dyspnoea"  # @param {type:"string"}
language.similarities([(PHRASE_1, PHRASE_2)])

## Part 3: search by meaning (13 min)

**Searching by meaning** (also known as semantic search) embeds all documents once and then embeds a specific query for every search to find the closest vectors.
The next cell embeds all 691 reports with our biomedical model.
As with notebook 01, any report that mentions ‘atrial fibrillation’ is considered an AF report.

In [ ]:
report_vectors = language.embed(reports, "biomedical")
is_af = np.array(["atrial fibrillation" in r.lower() for r in reports])
print(f"report_vectors: {report_vectors.shape[0]} reports, {report_vectors.shape[1]} numbers each")
print(f"AF reports: {is_af.sum()} of {len(reports)}")

### Scoring every report at once

You may recognise this step from the Multimodal lab. `report_vectors` is a table with a report on each row. `report_vectors @ query_vector` produces the dot product of every row in the table against the query vector and so we get a score for each report. An important consideration is that every single vector is length 1, which means that the score is equivalent to the cosine similarity.

### Finding the highest scores: `np.argsort`

`np.argsort(scores)` will not return the scores, but rather the **positions** of the scores ordered from lowest to highest.

In [ ]:
example = np.array([0.2, 0.9, 0.5])
print("np.argsort(example):         ", np.argsort(example))
print("reversed, with [::-1]:       ", np.argsort(example)[::-1])
print("first two, with [::-1][:2]:  ", np.argsort(example)[::-1][:2])

- 0.2 is the lowest score and is at position 0, so 0 is first: `[0 2 1]`.
- `[::-1]` inverts the order such that the highest is first: `[1 2 0]`.
- `[:2]` keeps the first two elements: `[1 2]` which are the positions of the two maximum values.

### TODO 2: the top k reports

Complete the two lines of `search`.

1. `scores`: finish the dot product with the query's vector, `query_vector`.
2. `top`: reverse the argsort, then keep the first `k`, as in the example above.

`show` runs a search and visualises the results in a table.

In [ ]:
def search(query, k=10):
    """The positions of the k reports closest in meaning to query (closest first), and every report's score."""
    query_vector = language.embed([query], "biomedical")[0]
    scores = report_vectors @ ...  # one cosine similarity per report
    top = np.argsort(scores)[...][...]  # reverse, then keep the first k
    return top, scores


def show(query, n=5, using=None):
    """Search, then show how many of the top 10 are AF reports and the first n reports."""
    language.show_search(search, query, reports, is_af, n, using)


if checks.search(search):
    show("heart rate too slow")

No report has text related to the term “too slow”, yet the top result is sinus bradycardia. The third result is sinus tachycardia, a heart rate that is too fast.

### TODO 3: three ways to ask for AF

`QUERIES` contains three different searches for AF reports. The first has been completed for you. Please add an abbreviation and a patient-based term.

In [ ]:
QUERIES = ["atrial fibrillation", ..., ...]
if checks.queries(QUERIES):
    for query in QUERIES:
        show(query, n=3)

### Where search by meaning fails

Three searches where search by meaning fails in a manner that is clinically relevant:

In [ ]:
for query in ["no atrial fibrillation", "heart rate 150", "heart beating irregularly"]:
    show(query, n=3)

- **Negation.** ‘No atrial fibrillation’ returns AF reports, and ‘Atrial fibrillation’ is the first result.
- **Numbers.** Searching for ‘heart rate 150’ returns sinus bradycardia as the top result. The embedding space is not necessarily numerically consistent; similarly, the reports are described based on rhythm and not rate.
- **Lay speech.** “Heart beating irregularly” finds no atrial fibrillation report in the top 10. Other rhythm descriptions are found instead.

Semantic search is a powerful tool for retrieving candidates that may have been paraphrased but is not a robust standalone filter. It is typically used in conjunction with keyword-based filtering or with a language model reviewing the candidates.

**3a.** Why does searching for “no atrial fibrillation” result in reports about AF?

- a) In `search` the scores are sorted in the wrong order.
- b) Positive and negated phrases remain close in the embedding space of this model and as such similarity search will rank them together.
- c) The biomedical model has never seen the words ‘atrial fibrillation’ during training.
- d) All reports in the data are about atrial fibrillation.

In [ ]:
check("3a", "")

### Try it: your own search

Type any search and run the cell. Try another negation, a number, a rhythm in lay terms, or a misspelling and see how the AF count and top reports change.

In [ ]:
MY_QUERY = "slow heart rate with a long PR interval"  # @param {type:"string"}
show(MY_QUERY)

## Part 4: predicting the next token (12 min)

We’ve discussed how an embedding model takes text and converts it into a single vector but a language model does something different. It provides a probability of the next token for all tokens in its vocabulary.

In [ ]:
prompt = "The ECG shows atrial"
language_figures.next_token_bars(lm.top_next_tokens(prompt, 6), prompt)

“ fibr” is the start of fibrillation and “ flutter” is the second option. The model has learnt from the text it has read which words typically follow.

### Try it: next-token probabilities

Type out the beginning of a sentence and then execute the cell. Try a partial drug name, a sentence that can have multiple endings, or one that only has a single ending.

In [ ]:
MY_PROMPT = "The patient was started on"  # @param {type:"string"}
language_figures.next_token_bars(lm.top_next_tokens(MY_PROMPT, 8), MY_PROMPT)

### From scores to probabilities: softmax

The final layer of the model provides a **score** (logit) for each of the 151,665 tokens. A score can be any arbitrary number, such as the score $z$ from notebook 01. The **softmax** function converts these to a probability distribution that sums to 1:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- $z_i$ is the score of the token $i$ and the sum below the line is over all tokens.
- $T$ is the **temperature**. For the moment this is 1.
- $e^{z}$ ensures all scores are positive and exaggerates the difference between large and small scores.
- The division by the sum ensures that the probabilities all add up to 1.

If there are only two options, with scores of $z$ and 0, this gives us $e^z / (e^z + 1) = 1 / (1 + e^{-z})$, i.e., the sigmoid function from notebook 01.

Worked example by hand for scores of 2, 1 and 0:

- exponentials: $e^2 = 7.39$, $e^1 = 2.72$, $e^0 = 1$
- the sum of these: $11.11$
- probabilities: $7.39 / 11.11 = 0.67$ followed by $0.24$ and $0.09$

### TODO 4: softmax

The first line is provided to you, which normalises the scores by subtracting the largest score from all the scores (this doesn’t change the probabilities but prevents the computer from overflowing a large number like $e^{1000}$) and then dividing by the temperature (which does change the probabilities but for now will be 1). Please fill in the two lines below.

- `e`: the exponential of every shifted score, with `np.exp`.
- The output is the division of each exponential by the sum of all exponentials, `e.sum()`.

When your check shows ✅, the cell will compare your softmax over all 151,665 scores with PyTorch's.

In [ ]:
def softmax(scores, temperature=1.0):
    """Turn a NumPy array of scores into probabilities that add up to 1, at the given temperature."""
    shifted = (scores - scores.max()) / temperature
    e = ...  # e to the power of each shifted score
    return ...  # each exponential divided by the sum of all of them


if checks.softmax(softmax):
    scores = lm.next_token_scores(prompt)
    library = torch.softmax(torch.tensor(scores), dim=0).numpy()
    print(f"Over all {len(scores):,} scores, yours and PyTorch's differ by at most "
          f"{np.abs(softmax(scores) - library).max():.0e}.")

### The generation loop

A reply is many tokens long, predicted one at a time.
**Generation** is a loop with the following steps:

1. Predict the probability of all potential next tokens.
2. Pick one.
3. Append this to the end of the text, then go back to step 1.

The simplest pick is merely selecting the most likely token (**greedy** picking).

### TODO 5: one step of the loop

`generate` begins `tokens` as the ids of the prompt. At each step `probs` has a probability for each token id. Fill in the two lines of code in the loop. The check tries your loop for three steps, and the cell then runs all 20.

- `next_token`: the position of the largest probability, `np.argmax(probs)`. That position is the token's id.
- We then append `next_token` to the end of `tokens` via `tokens.append(...)`.

In [ ]:
start = "The patient's ECG shows atrial fibrillation with"


def generate(steps=20):
    """Greedy generation from start: predict, pick the most likely token, append, repeat."""
    tokens = lm.tokens(start)
    for step in range(steps):
        probs = lm.next_token_probabilities(tokens)
        next_token = ...  # the position of the largest probability
        ...  # append next_token to tokens
    return tokens


if checks.generation(generate):
    tokens = generate()
    print("Prompt:     ", start)
    print("Model added:", lm.decode(tokens[len(lm.tokens(start)):]))

The model has **hallucinated** information: it has imagined a heart rate value that was never provided in the prompt and then converted the sentence into an exam-style question. It is following the pattern of what text like this typically continues with. Fluency is not truth.

### Sampling

Chat-based sites typically **sample** instead: they randomly select the next token with the probability given by the model. This means that you can have different answers given the same question.

### Temperature

Temperature scales all scores prior to the softmax calculation and only has an effect when sampling, as greedy selection will pick the most likely token irrespective of temperature.

- If less than 1, the differences between scores grow and so the probabilities sharpen.
- Above 1, the gaps narrow and the distribution is flattened.

### Try it: temperature

Try moving the slider and running the cell. You will see what the probabilities look like (computed with your `softmax`) and then 3 samples at that temperature. Try 0.1, 1, and 3. See how the three samples vary and how legible they are.

In [ ]:
TEMPERATURE = 1.0  # @param {type:"slider", min:0.1, max:3, step:0.1}
language_figures.temperature_bars(lm, start, softmax, temperatures=(TEMPERATURE,))
language.table({f"what the model added, sampled at temperature {TEMPERATURE:g}": lm.samples(start, TEMPERATURE)})

## Part 5: a conversation is one long text (5 min)

A conversation is just a list of **messages** with a role assigned to each:

- `system`: instructions from whoever built the app.
- `user`: what the person typed.
- `assistant`: the model's earlier replies.

The model still is just continuing text. There is a **chat template** that stitches all the messages together into a single block of text with markers around each message. The model was trained to continue this text by providing the response and then an end-of-turn token.

In [ ]:
messages = [{"role": "system", "content": "You explain ECG reports to patients in plain English, in one or two sentences."},
            {"role": "user", "content": "My ECG report says: Atrial fibrillation with rapid ventricular response. "
                                        "What does this mean?"}]
print(lm.chat_text(messages))

`lm.reply` runs the generation loop on this text using `generate` from the `transformers` library. The process terminates when the model predicts the end-of-turn token `<|im_end|>` or if it reaches `max_new_tokens`. If the response is cut off mid-sentence it has reached this limit.

The model has no memory between calls. It needs to be provided with the prior messages (or a summary of them) for it to answer follow-up questions. The next cell shows this by asking “Is it serious?” in isolation and in context.

In [ ]:
answer = lm.reply(messages, max_new_tokens=40)
follow_up = {"role": "user", "content": "Is it serious?"}
alone = lm.reply([{"role": "system", "content": "Answer briefly in plain English."}, follow_up], max_new_tokens=25)
with_history = lm.reply(messages + [{"role": "assistant", "content": answer}, follow_up], max_new_tokens=25)
language.table({"the model was sent": ["the question about the report", "\"Is it serious?\" on its own",
                                       "\"Is it serious?\" after the conversation"],
                "reply": [answer, alone, with_history]})

### Try it: the chat template

Try writing your own system message and question to see how the model reads the text and where your content is placed. Try a system message that asks for a single word or a response in a different language.

In [ ]:
MY_SYSTEM = "You are a cardiologist. Answer in one sentence."  # @param {type:"string"}
MY_QUESTION = "What is a normal resting heart rate?"  # @param {type:"string"}
mine = [{"role": "system", "content": MY_SYSTEM}, {"role": "user", "content": MY_QUESTION}]
print(lm.chat_text(mine))
print("Reply:", lm.reply(mine, max_new_tokens=40))

### A yes or no answer is one next token

A yes or no answer is just the first token in the response and so the probability can be read directly. The example below is the negated note from Part 3.

In [ ]:
question = [{"role": "system", "content": "Answer yes or no only."},
            {"role": "user", "content": "Note: No evidence of atrial fibrillation on today's ECG.\n"
                                        "Does this patient have atrial fibrillation?"}]
language.table(lm.top_next_tokens(lm.chat_text(question), 2), columns=["first token of the reply", "probability"])

On a clearly negative note, this small model has near equal probability of the next token being ‘No’ or ‘Yes’. If we were to greedily pick the next token, it would still select ‘No’ but only just. These numbers are not an error rate, nor are they a calibrated probability that the patient has AF. In the LLM integrations lab you will use a larger model via the standard API on clinic notes and measure how often it gets these types of questions correct.

### Take-home points

- **Text is broken into tokens**, from a set list of text fragments. Medical terms require several tokens. Model context lengths and costs are measured in tokens.
- **An embedding is a direction** and cosine similarity measures the closeness of two directions. It is sensitive to the words used and so can be broken by negation, numbers, and lay wording.
- **Language models produce probabilities** for every possible token that can follow, produced by applying softmax to the model's scores. These probabilities can be sharpened or blurred by modifying the temperature.
- **Generation is a loop** of predict-select-append; a chat is a single long string that a model repeatedly appends to until it predicts that it is finished.

## Extensions

These can be attempted in any order. The first three are more concerned with how the use of embeddings and tokens impacts the end users of the tool; the latter four look under the hood.

1. **Embedding bias.** What is similar to ‘nurse’? What about ‘surgeon’?
2. **Patients' own words.** What do patients actually write? Which lay terms succeed or fail?
3. **Other languages.** Does the same sentence cost more in another language?
4. **A search that understands ‘no’.** Except when it doesn’t.
5. **Maps of meaning.** Where do all 691 reports live, and how close are pairs of meanings?
6. **Sampling and top-k.** Sampling without the long tail of low probability tokens.
7. **Why `generate` is faster than your loop.** The key-value cache.

### Extension 1: bias in embeddings

Embedding models are used for everything from CV ranking, to referral matching, to clinical notes search. Embedding models learn from human text and so learn human biases as well. If a model is used to assist with ranking CVs for a position based on similarity to ‘surgeon’, these biases become decision points. This cell demonstrates how several words are embedded in relation to ‘nurse’ and ‘surgeon’ using the general model. Which words are closer to which, and where could this be a problem for healthcare?

In [ ]:
WORDS = ["she", "he", "woman", "man", "caring", "decisive", "gentle", "confident"]
vectors = language.embed(["nurse", "surgeon"] + WORDS, "general")
language.table({"word": WORDS, "nurse": (vectors[2:] @ vectors[0]).round(2),
                "surgeon": (vectors[2:] @ vectors[1]).round(2)})

### Extension 2: patients' own words

Patients do not search for ECG findings. They search for the symptoms they are experiencing. A search that only works in the language of the clinician fails patients. Try searching for “my heart is racing”, “I keep fainting”, or “slow pulse”, then try it with your own examples. Are the top results useful for a clinician and where does it fail?

In [ ]:
for query in ["my heart is racing", "I keep fainting", "slow pulse"]:
    show(query, n=3)

### Extension 3: tokens in other languages

For hosted models, you pay by the token and have a context window in tokens. If a sentence requires more tokens in another language, then a user of that language must pay more for the same answer and can fit less of their record into the context window. Compare the number of tokens required for the same sentence in four different languages with this model. Add a language you know!

In [ ]:
SENTENCES = {"English": "The patient has an irregular heartbeat and feels short of breath.",
             "French": "Le patient a un rythme cardiaque irrégulier et se sent essoufflé.",
             "Spanish": "El paciente tiene un latido cardíaco irregular y le falta el aire.",
             "Chinese": "患者心跳不规律，感到呼吸急促。"}
language.table([{"language": name, "tokens": len(lm.tokens(sentence)), "pieces": " | ".join(lm.pieces(sentence))}
                for name, sentence in SENTENCES.items()])

### Extension 4: a search that understands one kind of negation

Given a query such as ‘no atrial fibrillation’, one could simply search for the phrase ‘atrial fibrillation’ and exclude all reports that contain the phrase. The code cell below implements this logic. However, this logic is quite limited. What happens if you search for ‘no atrial flutter’ or ‘without AF’?

In [ ]:
def search_with_negation(query, k=10):
    """Search; for a query starting 'no ', return the closest reports that do not contain the rest of it."""
    if not query.lower().startswith("no "):
        return search(query, k)
    rest = query[3:].lower()
    order, scores = search(rest, len(reports))
    return np.array([i for i in order if rest not in reports[i].lower()][:k]), scores


show("no atrial fibrillation", using=search_with_negation)

### Extension 5: all the reports on one map, and pairs of meanings

This cell takes all reports and maps them by their biomedical vector, coloured by those that mention AF. This is a similar figure to the embedding map from notebook 01. What can you see about how the reports with AF cluster? What other types of reports are close to it?

In [ ]:
from adh_ai_labs import ecg

points = PCA(n_components=2).fit_transform(report_vectors)
ecg.plot_embedding(points, is_af, "The 691 reports, biomedical model, first two principal components")

#### How close are two meanings?

`language.similarities` embeds each pair of phrases with both models and then computes the cosine similarity between them. `language.embed` scales each vector to length 1, and so the cosine similarity is the same as the dot product `a @ b`.

In [ ]:
PAIRS = [("heart attack", "myocardial infarction"), ("heart attack", "MI"), ("atrial fibrillation", "AF"),
         ("paracetamol", "acetaminophen"), ("atrial fibrillation", "atrial flutter"),
         ("atrial fibrillation", "no atrial fibrillation"), ("heart rate 45", "heart rate 145"),
         ("atrial fibrillation", "broken leg")]
language.similarities(PAIRS)

- The biomedical model knows common abbreviations: “AF” is 0.82 similar to “atrial fibrillation” compared to 0.15 for the general model.
- A related rhythm, “Atrial flutter”, is as similar as a synonym.
- “No atrial fibrillation” scores 0.89, outscoring all synonyms, but means the opposite.
- ‘Heart rate 45’ and ‘heart rate 145’ have a similarity score of 0.72 and 0.90 for the general and biomedical models respectively despite the meaning being a slow vs a fast heart rate.

High similarity does not ensure the same clinical meaning, especially as it relates to negation and numbers.

### Extension 6: sampling and top-k sampling

Sometimes when the model samples a very unlikely token, the text will go off the rails.
**Top-k sampling** limits the tokens to sample from to only the `k` most likely tokens.
It does this by setting all other token probabilities to 0 and then redistributing the probability between the `k`.
Try running the cell below with `K = None` (standard sampling from all tokens), then try `K = 1`, `5`, and `50`.
Which one always produces the same text and why?

In [ ]:
K = 5
rng = np.random.default_rng()
continuations = []
for attempt in range(3):
    tokens = lm.tokens(start)
    for step in range(12):
        probs = lm.next_token_probabilities(tokens)
        keep = np.argsort(probs)[::-1][:K]  # the K most likely token ids (every token if K is None)
        tokens.append(int(rng.choice(keep, p=probs[keep] / probs[keep].sum())))
    continuations.append(lm.decode(tokens[len(lm.tokens(start)):]))
language.table({f"K = {K}": continuations})

### Extension 7: why `generate` is faster than your loop

Your loop re-processes all text at each step, whereas `generate` leverages the prior tokens’ computation (key-value cache) and only processes the new token. This cell times both methods for 20 tokens of greedy generation and verifies that they produce the same tokens. What would happen to each if you had to generate 500 tokens?

In [ ]:
import time

tokens = lm.tokens(start)
began = time.time()
for step in range(20):
    tokens.append(int(np.argmax(lm.next_token_probabilities(tokens))))
loop_seconds = time.time() - began
began = time.time()
cached = lm.greedy(lm.tokens(start), 20)
print(f"Your loop: {loop_seconds:.1f} s. generate: {time.time() - began:.1f} s. Same tokens: {cached == tokens}")

**Your answers:**